<a href="https://colab.research.google.com/github/Khushi545/AI-Powered-Workforce-Analytics-Talent-Intelligence-Dashboard/blob/main/sqlTask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Generate Dummy Blinkit Data

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# Create dummy data for blinkit_data
np.random.seed(42) # for reproducibility
data = {
    'order_id': range(1, 11),
    'user_id': np.random.randint(100, 200, 10),
    'product_id': np.random.randint(1000, 2000, 10),
    'product_name': [f'Product_{i}' for i in range(1, 11)],
    'category': np.random.choice(['Dairy & Breakfast', 'Fruits & Vegetables', 'Snacks & Beverages', 'Bakery & Cakes'], 10),
    'price': np.round(np.random.uniform(50, 500, 10), 2),
    'quantity': np.random.randint(1, 5, 10),
    'order_date': pd.to_datetime(pd.date_range(start='2023-01-01', periods=10, freq='D')),
    'delivery_status': np.random.choice(['Delivered', 'Pending', 'Cancelled'], 10),
    'rating': np.random.randint(1, 6, 10)
}

blinkit_df = pd.DataFrame(data)

# Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Write the DataFrame to a SQL table named 'blinkit_data'
blinkit_df.to_sql('blinkit_data', conn, index=False, if_exists='replace')

print("Dummy 'blinkit_data' created and loaded into an in-memory SQLite database.")

# Verify by querying the table
query = "SELECT * FROM blinkit_data LIMIT 5;"
result_df = pd.read_sql(query, conn)
display(result_df)

# Close the connection (optional, as it's in-memory and will be lost with script end)
# conn.close()

Dummy 'blinkit_data' created and loaded into an in-memory SQLite database.


,order_id,user_id,product_id,product_name,category,price,quantity,order_date,delivery_status,rating
0,1,151,1087,Product_1,Bakery & Cakes,286.14,3,2023-01-01 00:00:00,Cancelled,4
1,2,192,1372,Product_2,Fruits & Vegetables,244.38,4,2023-01-02 00:00:00,Pending,4
2,3,114,1099,Product_3,Fruits & Vegetables,181.05,4,2023-01-03 00:00:00,Delivered,4
3,4,171,1871,Product_4,Fruits & Vegetables,325.33,1,2023-01-04 00:00:00,Pending,4
4,5,160,1663,Product_5,Bakery & Cakes,112.77,3,2023-01-05 00:00:00,Pending,5


In [ ]:
union_all_query = """SELECT * FROM blinkit_data UNION ALL SELECT * FROM blinkit_data;"""

unioned_df = pd.read_sql(union_all_query, conn)
display(unioned_df.head())
print(f"The original blinkit_data has {len(blinkit_df)} rows.")
print(f"The UNION ALL result has {len(unioned_df)} rows.")

,order_id,user_id,product_id,product_name,category,price,quantity,order_date,delivery_status,rating
0,1,151,1087,Product_1,Bakery & Cakes,286.14,3,2023-01-01 00:00:00,Cancelled,4
1,2,192,1372,Product_2,Fruits & Vegetables,244.38,4,2023-01-02 00:00:00,Pending,4
2,3,114,1099,Product_3,Fruits & Vegetables,181.05,4,2023-01-03 00:00:00,Delivered,4
3,4,171,1871,Product_4,Fruits & Vegetables,325.33,1,2023-01-04 00:00:00,Pending,4
4,5,160,1663,Product_5,Bakery & Cakes,112.77,3,2023-01-05 00:00:00,Pending,5


The original blinkit_data has 10 rows.
The UNION ALL result has 20 rows.


In [ ]:
inner_join_query = """
SELECT
    b1.user_id,
    b1.order_id AS order_id_1,
    b1.product_name AS product_name_1,
    b2.order_id AS order_id_2,
    b2.product_name AS product_name_2
FROM
    blinkit_data AS b1
INNER JOIN
    blinkit_data AS b2
ON
    b1.user_id = b2.user_id AND b1.order_id < b2.order_id
ORDER BY
    b1.user_id, b1.order_id, b2.order_id;
"""

inner_joined_df = pd.read_sql(inner_join_query, conn)
display(inner_joined_df.head())
print(f"The INNER JOIN result has {len(inner_joined_df)} rows.")

,user_id,order_id_1,product_name_1,order_id_2,product_name_2
0,174,9,Product_9,10,Product_10


The INNER JOIN result has 1 rows.


This query joins the `blinkit_data` table with itself using aliases `b1` and `b2`. The join condition `b1.user_id = b2.user_id` ensures that we are looking at orders made by the same user. The `b1.order_id < b2.order_id` condition prevents duplicating pairs (e.g., A-B and B-A) and joining an order with itself.

In [ ]:
group_by_query = """
SELECT
    category,
    SUM(quantity) AS total_quantity_sold,
    AVG(price) AS average_price_per_item
FROM
    blinkit_data
GROUP BY
    category
ORDER BY
    total_quantity_sold DESC;
"""

grouped_df = pd.read_sql(group_by_query, conn)
display(grouped_df)
print(f"The GROUP BY result has {len(grouped_df)} rows.")

,category,total_quantity_sold,average_price_per_item
0,Fruits & Vegetables,10,222.6525
1,Bakery & Cakes,8,245.9275
2,Dairy & Breakfast,6,235.0450


The GROUP BY result has 3 rows.
